# Destilar la voz **Alex** (Kokoro) a Piper
Timbre de Alex, sin grabar nada. Guarda en tu Google Drive y **retoma si Colab se corta**.
Entrenamiento: fork `KiON-GiON/piper1-gpl@fixes` (comandos verificados contra el código real).

**Antes:** Entorno de ejecución → Cambiar tipo → **T4 GPU**.
**Celda 1** prepara · **Celda 2** entrena (reejecutable) · **Celda 3** exporta y descarga.

In [ ]:
#@title 1. PREPARAR — anti-desconexión, Google Drive, dataset e instalación
import IPython, torch, os
IPython.display.display(IPython.display.Javascript('function _keep(){ var b=document.querySelector("colab-toolbar-button#connect"); if(b) b.click() } setInterval(_keep, 60000)'))
assert torch.cuda.is_available(), "Activá T4: Entorno de ejecución -> Cambiar tipo de entorno -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
WORK = "/content/drive/MyDrive/LoudVox/alex"
os.makedirs(WORK, exist_ok=True)
print("Checkpoints en:", WORK)

# --- Generar el dataset con la voz Alex (Kokoro) ---
!pip install -q kokoro-onnx==0.5.0
!wget -q -nc -O /content/kokoro.onnx "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/kokoro-v1.0.onnx"
!wget -q -nc -O /content/voices.bin "https://github.com/thewh1teagle/kokoro-onnx/releases/download/model-files-v1.0/voices-v1.0.bin"
!wget -q -O /content/corpus.txt "https://raw.githubusercontent.com/Lazy-Money/Loud-Web/claude/readvox-research-slumjx/colab/corpus/corpus_es_1300.txt"
import wave, numpy as np
from kokoro_onnx import Kokoro
frases = [l.strip() for l in open("/content/corpus.txt", encoding="utf-8") if l.strip()]
print(f"{len(frases)} frases. Generando audio con Alex...")
kokoro = Kokoro("/content/kokoro.onnx", "/content/voices.bin")
os.makedirs("/content/dataset/wavs", exist_ok=True)
rows, total = [], 0.0
for i, f in enumerate(frases):
    try: s, r = kokoro.create(f, voice="em_alex", speed=1.0, lang="es")
    except Exception: continue
    idx = np.linspace(0, len(s)-1, int(len(s)*22050/r))
    d = np.clip(np.interp(idx, np.arange(len(s)), s)*32767, -32768, 32767).astype(np.int16)
    if not 1.0 <= len(d)/22050 <= 20.0: continue
    n = f"f{i:05d}.wav"
    with wave.open(f"/content/dataset/wavs/{n}","wb") as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(22050); w.writeframes(d.tobytes())
    rows.append(f"{n}|{f}"); total += len(d)/22050
    if len(rows)%200==0: print(f"  {len(rows)} frases...")
open("/content/dataset/metadata.csv","w",encoding="utf-8").write("\n".join(rows)+"\n")
print(f"Dataset: {len(rows)} clips, {total/60:.1f} min.")

print("Instalando Piper (fork KiON-GiON/piper1-gpl, verificado)...")
!apt-get -q update -y > /dev/null 2>&1
!apt-get -q install -y build-essential cmake ninja-build espeak-ng aria2 > /dev/null 2>&1
%cd /content
![ -d piper1-gpl ] || git clone -q -b fixes https://github.com/KiON-GiON/piper1-gpl.git
%cd /content/piper1-gpl
!python -m pip install -q -e .[train]
!bash build_monotonic_align.sh > /dev/null 2>&1
!pip install -q --upgrade gdown scikit-build protobuf==3.20.3
!python setup.py build_ext --inplace > /dev/null 2>&1
print("Descargando checkpoint base (espanol davefx) desde Hugging Face...")
from huggingface_hub import hf_hub_download
import shutil
_b = hf_hub_download(repo_id="rhasspy/piper-checkpoints", repo_type="dataset", filename="es/es_ES/davefx/medium/epoch=2218-step=562840.ckpt")
shutil.copy(_b, "/content/base.ckpt")
_mb = os.path.getsize("/content/base.ckpt") / 1e6
print(f"base.ckpt: {_mb:.1f} MB")
assert _mb > 10, "La descarga del checkpoint base fallo. Volve a correr la celda 1."
torch.load("/content/base.ckpt", map_location="cpu", weights_only=False)
print("Checkpoint base OK.")
print("\n=== LISTO. Corré la celda 2 para entrenar. ===")

In [ ]:
#@title 2. ENTRENAR — guarda en Drive cada 5 epochs; si Colab corta, reejecutá esta celda y retoma
import glob, os, re
WORK = "/content/drive/MyDrive/LoudVox/alex"
# Buscar un checkpoint previo en Drive (rankeado por version_N)
cks = glob.glob(WORK + "/lightning_logs/**/checkpoints/last.ckpt", recursive=True)
def _ver(p):
    m = re.search(r'version_(\d+)', p); return int(m.group(1)) if m else -1
if cks:
    prev = sorted(cks, key=_ver)[-1]
    resume = '--ckpt_path "' + prev + '"'
    print("RETOMANDO desde:", prev)
else:
    resume = '--model.init_from_checkpoint /content/base.ckpt'
    print("PRIMER entrenamiento: fine-tuning desde el checkpoint español.")
cmd = (
    "cd /content/piper1-gpl && python -m piper.train fit "
    '--data.voice_name "alex" '
    "--data.csv_path /content/dataset/metadata.csv "
    "--data.audio_dir /content/dataset/wavs "
    "--data.espeak_voice es "
    "--data.cache_dir /content/cache "
    '--data.config_path "' + WORK + '/alex.onnx.json" '
    "--data.batch_size 12 "
    "--model.sample_rate 22050 "
    "--data.validation_split 0 --data.num_test_examples 0 "
    '--trainer.default_root_dir "' + WORK + '" '
    "--trainer.accelerator gpu --trainer.devices 1 "
    "--trainer.max_epochs 10000 "
    "--trainer.precision 16-mixed "
    "--checkpoint.save_top_k 0 --checkpoint.monitor null "
    "--last_checkpoint.every_n_epochs 5 "
    + resume
)
print("\nEjecutando:\n", cmd, "\n")
get_ipython().system(cmd)

In [ ]:
#@title 3. EXPORTAR Y DESCARGAR — corré esto cuando quieras (aunque hayas cortado el entrenamiento)
import glob, os, re, json
WORK = "/content/drive/MyDrive/LoudVox/alex"
cks = glob.glob(WORK + "/lightning_logs/**/checkpoints/*.ckpt", recursive=True)
assert cks, "Todavía no hay checkpoint en Drive. Dejá correr la celda 2 unos minutos (guarda cada 5 epochs)."
def _ver(p):
    m = re.search(r'version_(\d+)', p); return int(m.group(1)) if m else -1
last = sorted(cks, key=_ver)[-1]
print("Exportando desde:", last)
# protobuf 3.20.3 se mantiene (el export a ONNX funciona igual, verificado)
get_ipython().system('cd /content/piper1-gpl && python -m piper.train.export_onnx --checkpoint "' + last + '" --output-file "' + WORK + '/alex.onnx"')
# asegurar phoneme_map en el config (compatibilidad con el motor)
cfgp = WORK + "/alex.onnx.json"
if os.path.exists(cfgp):
    cfg = json.load(open(cfgp, encoding="utf-8"))
    cfg.setdefault("phoneme_map", {})
    json.dump(cfg, open(cfgp, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
from google.colab import files
files.download(WORK + "/alex.onnx")
files.download(cfgp)
print("Copiá ambos a tu carpeta de voces de LoudVox y elegí 'alex' en Configuración")